# 01 - Dataset Split

This notebook creates the train, validation, and test split for the NIR-SORT dataset. The split is made at fabric level, so all measurements belonging to the same fabric stay in the same split.

All parseable microscopy and spectroscopy files are kept in the split output. Later notebooks choose the specific spectra or image records needed for a model.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import re
import shutil
import zipfile

import numpy as np
import pandas as pd

RAW_DATASET_ROOT_OVERRIDE = None

# Reproducible fabric-level split settings.
RANDOM_SEED = 42
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# CNN labels are used for classification and split strata. Composition percentages remain intact.
DOMINANCE_THRESHOLD = 50.0
MIN_CNN_CLASS_SAMPLES = 3

OVERWRITE_SPLITS = False
EXPECTED_ZIP_COUNT = 34

REQUIRED_RAW_ITEMS = ["README_main.txt", "Microscopy", "Spectroscopy", "full_dataset_composition_matrix.csv"]


def resolve_raw_dataset_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if all((candidate / item).exists() for item in REQUIRED_RAW_ITEMS):
            return candidate
    raise RuntimeError(
        "Could not resolve RAW_DATASET_ROOT. Set RAW_DATASET_ROOT_OVERRIDE to the folder "
        "containing README_main.txt, Microscopy/, Spectroscopy/, and full_dataset_composition_matrix.csv."
    )


def resolve_output_root(cwd: Path) -> Path:
    cwd = cwd.resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == "mdpp_v2":
            return candidate
        if (candidate / "mdpp_v2" / "notebooks").exists():
            return candidate / "mdpp_v2"
    return cwd / "mdpp_v2"


CWD = Path.cwd().resolve()
if RAW_DATASET_ROOT_OVERRIDE is None:
    RAW_DATASET_ROOT = resolve_raw_dataset_root(CWD)
else:
    override = Path(RAW_DATASET_ROOT_OVERRIDE).expanduser()
    if not override.is_absolute():
        override = CWD / override
    RAW_DATASET_ROOT = resolve_raw_dataset_root(override)

OUTPUT_ROOT = resolve_output_root(CWD)
ARTIFACTS = OUTPUT_ROOT / "artifacts"
DATA_SPLITS_DIR = OUTPUT_ROOT / "data_splits"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print("RAW_DATASET_ROOT:", RAW_DATASET_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DATA_SPLITS_DIR:", DATA_SPLITS_DIR)


## 1. Discover Source Files

The raw dataset is distributed as zip files. This step reads the zip contents, parses measurement filenames, and keeps one canonical record for each logical measurement.

In [ ]:
# NIR-SORT measurement filenames encode fabric, instrument, replicate, and data format.
# The parser is intentionally strict so helper/parameter CSV files are not treated as measurements.
FILENAME_RE = re.compile(
    r"^(?P<sample_id>[A-D]\d{2}_[a-z]\d{3}[a-z]\d{3}[a-z]\d{3}_\d{2}_\d{2}_)"
    r"(?P<instrument>[A-Z])_(?P<replicate>\d{2})(?P<data_format>[A-Z])\.[^.]+$",
    re.IGNORECASE,
)

# These are the measurement file types found inside the NIR-SORT microscopy/spectra zips.
ALLOWED_SUFFIXES = {".csv", ".bmp", ".tiff", ".tif"}

# Used only for duplicate resolution when the same logical measurement appears in two zips.
# Example: filename says A=Absorbance, so prefer the Absorbance directory over Reflectance.
FORMAT_DIR_BY_CODE = {
    "A": "Absorbance",
    "R": "Reflectance",
    "C": "Color",
    "M": "Mono",
    "P": "Polarized",
}

LOGICAL_MEASUREMENT_KEY = ["filename", "sample_id", "modality_group", "instrument", "data_format", "replicate"]

def parse_measurement_filename(filename: str) -> dict[str, str]:
    """Return NIR-SORT fields from one measurement filename."""
    match = FILENAME_RE.match(Path(filename).name)
    if not match:
        raise ValueError(f"Unrecognised NIR-SORT measurement filename: {filename}")
    fields = match.groupdict()
    fields["instrument"] = fields["instrument"].upper()
    fields["data_format"] = fields["data_format"].upper()
    fields["replicate"] = str(int(fields["replicate"]))
    return fields


def modality_from_zip_path(relative_zip_path: Path) -> tuple[str, str]:
    """Classify the parent zip path without filtering any supported modality out."""
    parts = relative_zip_path.parts
    if parts and parts[0] == "Microscopy":
        modality = parts[2] if len(parts) > 2 else "unknown"
        return "image", modality
    if parts and parts[0] == "Spectroscopy":
        modality = parts[2] if len(parts) > 2 else "unknown"
        return "spectra", modality
    return "unknown", "unknown"


def directory_matches_format(relative_zip_path: Path, data_format: str) -> bool:
    """True when the source folder agrees with the filename's data-format code."""
    expected = FORMAT_DIR_BY_CODE.get(data_format.upper())
    if expected is None:
        return True
    return expected in relative_zip_path.parts


def discover_zip_measurements(raw_root: Path):
    """Discover every parseable measurement in Microscopy and Spectroscopy zips."""
    zip_paths = sorted((raw_root / "Microscopy").rglob("*.zip")) + sorted((raw_root / "Spectroscopy").rglob("*.zip"))
    zip_rows = []
    raw_rows = []

    for zip_path in zip_paths:
        relative_zip_path = zip_path.relative_to(raw_root)
        stat = zip_path.stat()
        zip_rows.append(
            {
                "source_zip_relative_path": relative_zip_path.as_posix(),
                "source_zip_path": str(zip_path.resolve()),
                "size_bytes": stat.st_size,
                "mtime_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
            }
        )

        modality_group, modality = modality_from_zip_path(relative_zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                member_name = Path(info.filename).name
                if Path(member_name).suffix.lower() not in ALLOWED_SUFFIXES:
                    continue
                try:
                    fields = parse_measurement_filename(member_name)
                except ValueError:
                    continue
                raw_rows.append(
                    {
                        "source_zip_relative_path": relative_zip_path.as_posix(),
                        "source_zip_path": str(zip_path.resolve()),
                        "source_member": info.filename,
                        "filename": member_name,
                        "modality_group": modality_group,
                        "modality": modality,
                        "instrument": fields["instrument"],
                        "data_format": fields["data_format"],
                        "source": relative_zip_path.parent.name,
                        "sample_id": fields["sample_id"],
                        "replicate": fields["replicate"],
                        "format_dir_match": directory_matches_format(relative_zip_path, fields["data_format"]),
                    }
                )

    zip_manifest = pd.DataFrame(zip_rows)
    raw_manifest = pd.DataFrame(raw_rows)
 
    if raw_manifest.empty:
        raise RuntimeError("No parseable measurement files found inside source zips.")

    raw_manifest = raw_manifest.reset_index(drop=True).reset_index(names="raw_row_id")
 
    sorted_manifest = raw_manifest.assign(format_rank=np.where(raw_manifest["format_dir_match"], 0, 1)).sort_values(
        LOGICAL_MEASUREMENT_KEY + ["format_rank", "source_zip_relative_path", "source_member"]
    )
 
    canonical_manifest = sorted_manifest.drop_duplicates(LOGICAL_MEASUREMENT_KEY, keep="first").drop(columns=["format_rank"])
    skipped_duplicates = raw_manifest[~raw_manifest["raw_row_id"].isin(set(canonical_manifest["raw_row_id"]))].copy()
 
    canonical_manifest = canonical_manifest.drop(columns=["raw_row_id"]).sort_values(
        ["sample_id", "modality_group", "instrument", "data_format", "replicate", "filename", "source_zip_relative_path"]
    ).reset_index(drop=True)

    return zip_manifest, raw_manifest, canonical_manifest, skipped_duplicates


source_zip_manifest, raw_measurement_manifest, canonical_manifest, skipped_duplicate_measurements = discover_zip_measurements(
    RAW_DATASET_ROOT
)

source_zip_manifest.to_csv(ARTIFACTS / "source_zip_manifest.csv", index=False)


print("source zips:", len(source_zip_manifest))
print("raw parsed zip measurements:", len(raw_measurement_manifest))
print("canonical measurements:", len(canonical_manifest))
print("skipped duplicate measurements:", len(skipped_duplicate_measurements))
print("unique fabrics:", canonical_manifest["sample_id"].nunique())

modality_test_type_counts = canonical_manifest.groupby(
    ["modality_group", "modality", "instrument", "data_format"]
).size().rename("files").reset_index()


print("modality/test-type combinations kept:", len(modality_test_type_counts))

discovery_summary = pd.DataFrame([
    {"quantity": "source zip files", "value": len(source_zip_manifest)},
    {"quantity": "raw parsed measurements", "value": len(raw_measurement_manifest)},
    {"quantity": "canonical measurements", "value": len(canonical_manifest)},
    {"quantity": "skipped duplicate measurements", "value": len(skipped_duplicate_measurements)},
    {"quantity": "unique fabrics", "value": canonical_manifest["sample_id"].nunique()},
    {"quantity": "modality/test-type combinations", "value": len(modality_test_type_counts)},
])
discovery_summary


## 2. Build Composition and CNN Labels

Composition percentages are kept for retrieval and composition evaluation. CNN labels are derived from fabrics with one dominant fibre class.

In [ ]:
FIBRE_COLUMNS = [
    "Acrylic",
    "Cotton",
    "Natural polyamide",
    "Polyester / recycled polyester / PET",
    "Regenerated cellulosic",
    "Spandex",
    "Synthetic polyamide / nylon",
]

EXCLUDED_CNN_LABEL = "EXCLUDED_CNN"


def composition_vectors(composition_table: pd.DataFrame) -> pd.DataFrame:
    """Keep the full percentage vector for retrieval-quality and fusion evaluation."""
    cols = ["Sample ID"] + [c for c in FIBRE_COLUMNS if c in composition_table.columns]
    out = composition_table[cols].copy()
    for col in cols[1:]:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0.0)
    return out.set_index("Sample ID")


def primary_fibre_label(row: pd.Series) -> str:
    """Largest fibre percentage, used only for descriptive audit context."""
    values = row[[c for c in FIBRE_COLUMNS if c in row.index]].astype(float)
    if values.sum() <= 0:
        return "unknown"
    return str(values.idxmax())


def composition_code_label(row: pd.Series) -> str:
    """Exact blend/code label from the metadata, useful for audit and reports."""
    if "Composition Code" in row and pd.notna(row["Composition Code"]):
        return str(row["Composition Code"])
    parts = []
    for fibre in FIBRE_COLUMNS:
        if fibre in row and float(row[fibre]) > 0:
            parts.append(f"{fibre}:{float(row[fibre]):g}")
    return "|".join(parts) if parts else "unknown"


def make_dominant_fibre_labels(
    composition_table: pd.DataFrame,
    dominance_threshold: float,
    min_samples_per_class: int,
    excluded_label: str = EXCLUDED_CNN_LABEL,
) -> pd.DataFrame:
    """Create CNN classification labels and a split stratum without retrieval proxies."""
    rows = composition_table.copy()
    rows["exact_label"] = rows.apply(composition_code_label, axis=1)
    rows["primary_label"] = rows.apply(primary_fibre_label, axis=1)

    # First pass: detect whether each fabric has exactly one fibre over the threshold.
    dominant_fibres = []
    dominant_percentages = []
    dominance_reasons = []

    for _, row in rows.iterrows():
        values = row[[c for c in FIBRE_COLUMNS if c in row.index]].astype(float)
        max_value = float(values.max())
        winners = values[values == max_value].index.tolist()
        dominant_percentages.append(max_value)
        if max_value > dominance_threshold and len(winners) == 1:
            dominant_fibres.append(winners[0])
            dominance_reasons.append("")
        elif len(winners) > 1:
            dominant_fibres.append(None)
            dominance_reasons.append("tie")
        else:
            dominant_fibres.append(None)
            dominance_reasons.append("no_fibre_above_threshold")

    rows["dominant_fibre"] = dominant_fibres
    rows["dominant_percent"] = dominant_percentages
    rows["cnn_exclusion_reason"] = dominance_reasons

    # Second pass: remove dominant classes too small for train/val/test coverage.
    eligible_counts = rows["dominant_fibre"].dropna().value_counts()
    too_small = rows["dominant_fibre"].map(lambda label: pd.notna(label) and eligible_counts[label] < min_samples_per_class)
    rows.loc[too_small, "cnn_exclusion_reason"] = "class_has_fewer_than_min_samples"

    rows["cnn_eligible"] = rows["cnn_exclusion_reason"].eq("")
    rows["cnn_label"] = rows["dominant_fibre"].where(rows["cnn_eligible"], excluded_label)
    rows["classification_label"] = rows["cnn_label"].where(rows["cnn_eligible"], pd.NA)
    rows["split_stratum"] = rows["cnn_label"]

    return rows[
        [
            "Sample ID",
            "exact_label",
            "primary_label",
            "dominant_fibre",
            "dominant_percent",
            "cnn_label",
            "cnn_eligible",
            "cnn_exclusion_reason",
            "classification_label",
            "split_stratum",
        ]
    ]


composition_table = pd.read_csv(RAW_DATASET_ROOT / "full_dataset_composition_matrix.csv")
composition = composition_vectors(composition_table)
# Cache full fibre percentage vectors in mdpp_v2 so later notebooks do not need the raw root.
composition.reset_index().to_csv(ARTIFACTS / "composition_vectors.csv", index=False)
class_table = make_dominant_fibre_labels(
    composition_table,
    dominance_threshold=DOMINANCE_THRESHOLD,
    min_samples_per_class=MIN_CNN_CLASS_SAMPLES,
)

manifest_samples = set(canonical_manifest["sample_id"].unique())
composition_samples = set(class_table["Sample ID"].unique())
missing_in_manifest = sorted(composition_samples - manifest_samples)
missing_in_composition = sorted(manifest_samples - composition_samples)

print(f"CNN policy: one fibre > {DOMINANCE_THRESHOLD:g}%, class count >= {MIN_CNN_CLASS_SAMPLES}")
print("composition rows:", len(composition_table))
print("CNN eligible:", int(class_table["cnn_eligible"].sum()))
print("CNN excluded:", int((~class_table["cnn_eligible"]).sum()))
cnn_label_summary = class_table["cnn_label"].value_counts().rename_axis("cnn_label").reset_index(name="samples")
cnn_label_summary


## 3. Make Fabric-Level Splits

The split unit is the fabric ID. This prevents images, spectra, or repeated measurements from the same fabric appearing in more than one split.

In [ ]:
def sample_level_split(
    class_table: pd.DataFrame,
    train_size: float,
    val_size: float,
    test_size: float,
    seed: int,
) -> pd.DataFrame:
    """Split fabrics by class while keeping every replicate/modality together."""
    if round(train_size + val_size + test_size, 6) != 1.0:
        raise ValueError("Split sizes must sum to 1")

    rng = np.random.default_rng(seed)
    samples = class_table.drop_duplicates("Sample ID").reset_index(drop=True)
    split_rows = []

    # Split independently inside each class so validation/test retain class coverage.
    for _, group in samples.groupby("split_stratum", sort=True):
        indices = group.index.to_numpy().copy()
        rng.shuffle(indices)
        shuffled = group.loc[indices].reset_index(drop=True)
        n = len(shuffled)

        # Small classes need special handling; otherwise validation/test can erase train coverage.
        if n == 1:
            counts = {"train": 1, "val": 0, "test": 0}
        elif n == 2:
            counts = {"train": 1, "val": 1, "test": 0}
        else:
            n_val = max(1, int(round(n * val_size)))
            n_test = max(1, int(round(n * test_size)))
            while n - n_val - n_test < 1:
                if n_val >= n_test and n_val > 1:
                    n_val -= 1
                elif n_test > 1:
                    n_test -= 1
                else:
                    break
            counts = {"train": n - n_val - n_test, "val": n_val, "test": n_test}

        start = 0
        for split_name in ("train", "val", "test"):
            count = counts[split_name]
            if count == 0:
                continue
            split_rows.append(shuffled.iloc[start : start + count].assign(split=split_name))
            start += count

    return pd.concat(split_rows, ignore_index=True)


def check_no_sample_leakage(split_table: pd.DataFrame) -> None:
    """Fail if any fabric appears in more than one split."""
    counts = split_table.groupby("Sample ID")["split"].nunique()
    leaking = counts[counts > 1]
    if not leaking.empty:
        raise ValueError(f"Sample leakage detected: {leaking.index.tolist()}")


def stratification_report(split_table: pd.DataFrame) -> pd.DataFrame:
    """Class-by-split fabric counts."""
    return (
        split_table.pivot_table(index="split_stratum", columns="split", values="Sample ID", aggfunc="count", fill_value=0)
        .reindex(columns=["train", "val", "test"], fill_value=0)
        .astype(int)
    )


sample_splits = sample_level_split(
    class_table,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
    seed=RANDOM_SEED,
)
check_no_sample_leakage(sample_splits)
split_counts = sample_splits["split"].value_counts().reindex(["train", "val", "test"]).astype(int)
sample_splits.to_csv(ARTIFACTS / "sample_splits.csv", index=False)
strat_report = stratification_report(sample_splits)

visible_split_counts = split_counts.drop(index="test")
visible_strat_report = strat_report[["train", "val"]]

print("Train/validation split counts:", visible_split_counts.to_dict())
visible_strat_report


## 4. Extract Split Files

This writes the split folders under `data_splits/`. Existing split files are reused unless `OVERWRITE_SPLITS = True` is set in the first code cell.

In [ ]:
def final_relative_path_for_row(row: pd.Series) -> str:
    """Preserve raw modality/source hierarchy inside the chosen split folder."""
    source_parent = Path(row["source_zip_relative_path"]).parent
    return (Path("data_splits") / row["split"] / source_parent / row["filename"]).as_posix()


def copy_zip_members_for_group(zip_path: Path, rows: pd.DataFrame) -> None:
    """Extract all selected members from one zip before moving to the next zip."""
    with zipfile.ZipFile(zip_path) as zf:
        for _, row in rows.iterrows():
            destination = Path(row["final_path"])
            destination.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(row["source_member"]) as src, destination.open("wb") as dst:
                shutil.copyfileobj(src, dst)


# Attach the fabric-level split to every measurement. No modality filtering happens here.
split_lookup = sample_splits.set_index("Sample ID")["split"].to_dict()
expected_extracted_manifest = canonical_manifest.copy()
expected_extracted_manifest["split"] = expected_extracted_manifest["sample_id"].map(split_lookup)
missing_split = expected_extracted_manifest[expected_extracted_manifest["split"].isna()]["sample_id"].drop_duplicates().tolist()
expected_extracted_manifest["final_relative_path"] = expected_extracted_manifest.apply(final_relative_path_for_row, axis=1)
expected_extracted_manifest["final_path"] = expected_extracted_manifest["final_relative_path"].map(lambda p: str((OUTPUT_ROOT / p).resolve()))

# Since path context is preserved, any duplicate final path means a real collision bug.
path_duplicates = expected_extracted_manifest[expected_extracted_manifest["final_relative_path"].duplicated(keep=False)]

existing_split_data = DATA_SPLITS_DIR.exists() and any(DATA_SPLITS_DIR.iterdir())
lock_path = DATA_SPLITS_DIR / "test" / "README_LOCKED.txt"
manifest_path = ARTIFACTS / "extracted_file_manifest.csv"

if existing_split_data and not OVERWRITE_SPLITS:
    print(f"{DATA_SPLITS_DIR} already contains files; reusing existing extracted manifests.")
    if not manifest_path.exists():
        raise RuntimeError(f"Existing split data found, but {manifest_path} is missing.")
    extracted_manifest = pd.read_csv(manifest_path)

    key_cols = ["source_zip_relative_path", "source_member", "sample_id"]
    missing_columns = sorted(set(expected_extracted_manifest.columns) - set(extracted_manifest.columns))
    expected_keys = set(map(tuple, expected_extracted_manifest[key_cols].astype(str).to_numpy()))
    existing_keys = set(map(tuple, extracted_manifest[key_cols].astype(str).to_numpy()))
    if expected_keys != existing_keys:
        raise RuntimeError("Existing extracted manifest does not match current zip discovery. Set OVERWRITE_SPLITS=True to regenerate.")

    expected_split_by_sample = expected_extracted_manifest.drop_duplicates("sample_id").set_index("sample_id")["split"]
    existing_split_by_sample = extracted_manifest.drop_duplicates("sample_id").set_index("sample_id")["split"]
    changed_split_samples = sorted(
        sample_id for sample_id in expected_split_by_sample.index
        if existing_split_by_sample.get(sample_id) != expected_split_by_sample.loc[sample_id]
    )
    if changed_split_samples:
        raise RuntimeError(
            "Existing data_splits use different split assignments. "
            f"Set OVERWRITE_SPLITS=True to regenerate. Examples: {changed_split_samples[:10]}"
        )
else:
    if DATA_SPLITS_DIR.exists() and OVERWRITE_SPLITS:
        shutil.rmtree(DATA_SPLITS_DIR)
    DATA_SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    extracted_manifest = expected_extracted_manifest

    # Extract every canonical measurement, including modalities not used by the first model.
    for source_zip_path, zip_rows in extracted_manifest.groupby("source_zip_path", sort=True):
        copy_zip_members_for_group(Path(source_zip_path), zip_rows)

    for split_name in ["train", "val", "test"]:
        split_dir = DATA_SPLITS_DIR / split_name
        split_dir.mkdir(parents=True, exist_ok=True)
        extracted_manifest[extracted_manifest["split"].eq(split_name)].to_csv(split_dir / "measurements_manifest.csv", index=False)

    lock_path.write_text(
        "Held-out test split. Do not use for training, model selection, vector-store building, or development metrics.\n",
        encoding="utf-8",
    )
    extracted_manifest.to_csv(manifest_path, index=False)

# Always refresh sample-level CSVs. Label schema changes do not require file extraction.
for split_name in ["train", "val", "test"]:
    split_dir = DATA_SPLITS_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    sample_splits[sample_splits["split"].eq(split_name)].to_csv(split_dir / "samples.csv", index=False)

print("Extracted measurement files:", len(extracted_manifest))


In [ ]:
split_sample_sets = {
    split_name: set(sample_splits.loc[sample_splits["split"].eq(split_name), "Sample ID"])
    for split_name in ["train", "val", "test"]
}


sample_split_counts = extracted_manifest.groupby("sample_id")["split"].nunique()

na_samples = set(extracted_manifest.loc[
    extracted_manifest["instrument"].eq("N") & extracted_manifest["data_format"].eq("A"),
    "sample_id",
])
missing_na_by_split = {
    split_name: sorted(samples - na_samples)
    for split_name, samples in split_sample_sets.items()
}

missing_paths = [p for p in extracted_manifest["final_path"] if not Path(p).exists()]

measurement_output_files = [
    p for p in DATA_SPLITS_DIR.rglob("*")
    if p.is_file() and p.name not in {"samples.csv", "measurements_manifest.csv", "README_LOCKED.txt"}
]

split_counts = sample_splits["split"].value_counts().reindex(["train", "val", "test"]).astype(int)

print("Split summary:")
print(f"Train/validation fabrics: {int(split_counts['train'])}/{int(split_counts['val'])}")
print(f"Extracted measurement files: {len(extracted_manifest)}")
